In [1]:
import sys, json
sys.path.insert(0, r'C:\Users\tglaubach\repos\pydocmaker\src')

## Writing Word docx Documents with templates and fields

Below is an example on how to use pydocmaker to write word docx documents from format templates
and also automatically "replace" fields (MergeFields in Word or plain text) to be filled out in the docx document with text from python.

**NOTE**: Updating word documents and exporting them to PDF requires the `win32com` api, Microsoft Word installed and only works on Windows.

**NOTE**: Exporting word documents is unfortunately very slow, but hey... it works :-)




In [2]:

import pydocmaker as pyd
import os


### Prepare a Word template file

prepare a report, a template and some fields in the template:

In [3]:
templatepath = os.path.join(pyd.get_registered_template_dirs()[0], 'word_template_with_mergefields.docx')
templatepath, os.path.exists(templatepath)

('C:\\Users\\tglaubach\\repos\\pydocmaker\\src\\pydocmaker\\templates\\word_template_with_mergefields.docx',
 True)

In [4]:
# HOWTO: 
#  Adding MergeFields In Word to replace them later: 
#    Go to Insert -> Quick Parts -> Field -> MergeField.

metadata = {
    'repno': "1234",
    "summary": "This is a nice workflow for automatically creating docx documents",
    "date": "2025-12-13",
    "comment": f"this is my comment!",
    "author": "Me"
}


# get a pyd example document to show the concept
doc = pyd.get_example()

this is the quick and easy way using the common pydocmaker api:

### To Word docx (without additional dependencies)

In [5]:
# three different examples below
file_exists = doc.to_docx("any_path_to_my_outfile.docx", template=templatepath, template_params=metadata, use_w32=False)
file_exists


True

### To Word docx (with additional dependencies)

This will also update all fields and the table of contents, but will need win32com for it.

In [6]:
file_exists = doc.to_docx("any_path_to_my_outfile_w32.docx", template=templatepath, template_params=metadata, use_w32=True)
file_exists

True

### To PDF via Word docx (with additional dependencies)

This will do the same as ion the last step, but also export the file to PDF after it has created the `.docx` file. 

In [7]:
file_exists = doc.to_docx("any_path_to_my_outfile_w32_comp.pdf", template=templatepath, template_params=metadata, use_w32=True, as_pdf=True, compress_images=True)
file_exists

True

### Manual export using the low level classes and functions

you can also work with the exporting classes directly to get more control:

In [8]:
outpath = 'any_path_to_my_outfile2.docx'
outpath_pdf = outpath.replace(".docx", ".pdf")

docxf = pyd.DocxFile(templatepath).replace_fields(metadata).append(doc.to_docx())
docxf.save(outpath)

if pyd.DocxFileW32.is_installed():
    with pyd.DocxFileW32(outpath) as docxw32f:
        docxw32f.update_fields()
        docxw32f.compress_images()
        docxw32f.export( outpath_pdf )

os.path.exists(outpath), os.path.exists(outpath_pdf)

(True, True)